# DVN AT3 Integration Notebook

This notebook integrates cleaned BITRE, ABS population, and Open-Meteo weather datasets
into Tableau-ready files. The key principle is to preserve BITRE row-level fatality
records while adding population and weather context with validated join keys.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


## 1. File Paths

Update these paths if running in Colab. In Colab, upload the cleaned CSVs and use
paths such as `/content/bitre_clean_detail.csv`.


In [ ]:
BASE_DIR = Path("/Users/shameelzeshan/Desktop/SEM 2/DVN/assignment 3/Cleaned dataset")

BITRE_DETAIL_PATH = BASE_DIR / "bitre_clean_outputs" / "bitre_clean_detail.csv"
BITRE_STATE_MONTH_PATH = BASE_DIR / "bitre_clean_outputs" / "bitre_state_month.csv"
POPULATION_PATH = BASE_DIR / "population_state_year_clean.csv"
WEATHER_PATH = BASE_DIR / "weather_monthly_state_clean_2024_jan2026.csv"

OUTPUT_DIR = BASE_DIR / "integrated_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load Cleaned Datasets


In [ ]:
bitre = pd.read_csv(BITRE_DETAIL_PATH)
bitre_state_month = pd.read_csv(BITRE_STATE_MONTH_PATH)
population = pd.read_csv(POPULATION_PATH)
weather = pd.read_csv(WEATHER_PATH)

print("Loaded datasets")
print("BITRE detail:", bitre.shape)
print("BITRE state-month:", bitre_state_month.shape)
print("Population:", population.shape)
print("Weather:", weather.shape)


## 3. Standardise Join Key Types


In [ ]:
for df in [bitre, bitre_state_month, population, weather]:
    if "state" in df.columns:
        df["state"] = df["state"].astype(str).str.strip().str.upper()
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    if "month" in df.columns:
        df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")

bitre["month_start"] = pd.to_datetime(bitre["month_start"], errors="coerce")
bitre_state_month["month_start"] = pd.to_datetime(bitre_state_month["month_start"], errors="coerce")
population["quarter_date"] = pd.to_datetime(population["quarter_date"], dayfirst=True, errors="coerce")

print("Join key standardisation complete")


## 4. Validate Join Keys Before Integration


In [ ]:
print("Population duplicate state-year keys:")
print(population.duplicated(subset=["state", "year"]).sum())

print("\nWeather duplicate state-year-month keys:")
print(weather.duplicated(subset=["state", "year", "month"]).sum())

print("\nBITRE states:", sorted(bitre["state"].dropna().unique()))
print("Population states:", sorted(population["state"].dropna().unique()))
print("Weather states:", sorted(weather["state"].dropna().unique()))

print("\nBITRE year range:", bitre["year"].min(), "to", bitre["year"].max())
print("Population year range:", population["year"].min(), "to", population["year"].max())
print("Weather year-month range:", weather["year_month"].min(), "to", weather["year_month"].max())


## 5. Integrate BITRE Detail + Population + Weather

Population is annual and joins on `state + year`.
Weather is monthly and joins on `state + year + month`.
Weather is contextual only, not exact crash-day weather.


In [ ]:
master = bitre.copy()
original_rows = len(master)

population_for_join = population.rename(
    columns={
        "quarter_date": "population_quarter_date",
        "state_name": "population_state_name",
    }
)

master = master.merge(
    population_for_join,
    on=["state", "year"],
    how="left",
    validate="many_to_one",
)

master["population_available"] = np.where(master["population"].notna(), "Yes", "No")

weather_for_join = weather.rename(
    columns={
        "city": "weather_city",
        "year_month": "weather_year_month",
    }
)

master = master.merge(
    weather_for_join,
    on=["state", "year", "month"],
    how="left",
    validate="many_to_one",
)

master["weather_available"] = np.where(master["weather_city"].notna(), "Yes", "No")

master["rate_analysis_window"] = np.where(
    (master["population"].notna()) & (master["year"] <= 2025),
    "Population rate available",
    "Population rate unavailable",
)

master["weather_analysis_window"] = np.where(
    master["weather_available"].eq("Yes"),
    "Weather context available",
    "Weather context unavailable",
)

print("Master row count before:", original_rows)
print("Master row count after:", len(master))
print("Rows changed:", len(master) - original_rows)

print("\nPopulation availability:")
print(master["population_available"].value_counts(dropna=False))

print("\nWeather availability:")
print(master["weather_available"].value_counts(dropna=False))


## 6. Build State-Year Risk Rates

This is the safest file for the HD population-adjusted risk visual.


In [ ]:
state_year_rates = (
    bitre.groupby(["state", "year"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
)

state_year_rates = state_year_rates.merge(
    population_for_join,
    on=["state", "year"],
    how="left",
    validate="many_to_one",
)

state_year_rates["fatalities_per_100k"] = (
    state_year_rates["fatalities"] / state_year_rates["population"] * 100000
)

state_year_rates["population_available"] = np.where(
    state_year_rates["population"].notna(),
    "Yes",
    "No",
)

state_year_rates["analysis_window"] = np.select(
    [
        state_year_rates["year"].between(1989, 2023),
        state_year_rates["year"].between(2024, 2025),
        state_year_rates["year"].eq(2026),
    ],
    [
        "Historical: 1989-2023",
        "Full recent years: 2024-2025",
        "Partial latest year: Jan 2026",
    ],
    default="Other",
)

state_year_rates_recent_full = state_year_rates[
    state_year_rates["year"].isin([2024, 2025])
].copy()

print("State-year rates:", state_year_rates.shape)
print("Missing population rows:", state_year_rates["population"].isna().sum())
print("\nLatest rate years:")
print(state_year_rates[state_year_rates["year"] >= 2024].head(20).to_string(index=False))


## 7. Build State-Month Dashboard File

This keeps the historical trend file small while adding population/weather context where available.


In [ ]:
state_month_dashboard = bitre_state_month.copy()

state_month_dashboard = state_month_dashboard.merge(
    population_for_join,
    on=["state", "year"],
    how="left",
    validate="many_to_one",
)

state_month_dashboard = state_month_dashboard.merge(
    weather_for_join,
    on=["state", "year", "month"],
    how="left",
    validate="many_to_one",
)

state_month_dashboard["fatalities_per_100k_month"] = (
    state_month_dashboard["fatalities"] / state_month_dashboard["population"] * 100000
)

state_month_dashboard["population_available"] = np.where(
    state_month_dashboard["population"].notna(),
    "Yes",
    "No",
)

state_month_dashboard["weather_available"] = np.where(
    state_month_dashboard["weather_city"].notna(),
    "Yes",
    "No",
)

print("State-month dashboard:", state_month_dashboard.shape)
print("Population availability:")
print(state_month_dashboard["population_available"].value_counts(dropna=False))
print("Weather availability:")
print(state_month_dashboard["weather_available"].value_counts(dropna=False))


## 8. Build Risk Segment EDA Tables

These support the dashboard's pattern-finder and what-if lives-saved panel.


In [ ]:
latest = master[master["analysis_window"].eq("Recent: 2024-Jan 2026")].copy()
recent_full_years = master[master["year"].isin([2024, 2025])].copy()

total_latest_fatalities = latest["deaths"].sum()

risk_segments_latest = (
    latest.groupby(
        ["state", "remoteness_group", "speed_band", "road_user_group"],
        dropna=False,
        as_index=False,
    )
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

risk_segments_latest["share_of_latest_fatalities"] = (
    risk_segments_latest["fatalities"] / total_latest_fatalities
)

day_time_heatmap_latest = (
    latest.groupby(["dayweek", "time_band"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

road_user_speed_latest = (
    latest.groupby(["road_user_group", "speed_band"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

remoteness_speed_latest = (
    latest.groupby(["remoteness_group", "speed_band"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

print("Top latest risk segments:")
print(risk_segments_latest.head(10).to_string(index=False))

print("\nTop day-time cells:")
print(day_time_heatmap_latest.head(10).to_string(index=False))


## 9. Create EDA Key Findings Table


In [ ]:
annual_fatalities = (
    bitre.groupby("year", as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("year")
)

latest_fatalities = int(latest["deaths"].sum())
fatalities_2024 = int(master.loc[master["year"].eq(2024), "deaths"].sum())
fatalities_2025 = int(master.loc[master["year"].eq(2025), "deaths"].sum())
fatalities_jan2026 = int(master.loc[master["year"].eq(2026), "deaths"].sum())

highest_recent_risk_state = (
    state_year_rates_recent_full.groupby("state", as_index=False)
    .agg(
        fatalities=("fatalities", "sum"),
        avg_population=("population", "mean"),
    )
)
highest_recent_risk_state["fatalities_per_100k"] = (
    highest_recent_risk_state["fatalities"] / highest_recent_risk_state["avg_population"] * 100000
)
highest_recent_risk_state = highest_recent_risk_state.sort_values(
    "fatalities_per_100k",
    ascending=False,
).iloc[0]

top_segment = risk_segments_latest.iloc[0]
top_day_time = day_time_heatmap_latest.iloc[0]

eda_key_findings = pd.DataFrame(
    [
        {
            "finding": "Total historical fatalities",
            "value": int(master["deaths"].sum()),
            "period": "1989-Jan 2026",
            "dashboard_use": "KPI card and historical scale",
        },
        {
            "finding": "Latest-window fatalities",
            "value": latest_fatalities,
            "period": "2024-Jan 2026",
            "dashboard_use": "Latest warning KPI",
        },
        {
            "finding": "2024 fatalities",
            "value": fatalities_2024,
            "period": "2024",
            "dashboard_use": "Full recent year comparison",
        },
        {
            "finding": "2025 fatalities",
            "value": fatalities_2025,
            "period": "2025",
            "dashboard_use": "Full recent year comparison",
        },
        {
            "finding": "January 2026 fatalities",
            "value": fatalities_jan2026,
            "period": "Jan 2026",
            "dashboard_use": "Latest available month context",
        },
        {
            "finding": "Highest recent population-adjusted risk state",
            "value": highest_recent_risk_state["state"],
            "period": "2024-2025",
            "dashboard_use": "Fatalities per 100,000 ranked bar",
        },
        {
            "finding": "Highest recent risk state's fatalities per 100k",
            "value": round(float(highest_recent_risk_state["fatalities_per_100k"]), 2),
            "period": "2024-2025",
            "dashboard_use": "Highest risk KPI",
        },
        {
            "finding": "Top latest risk segment",
            "value": (
                f"{top_segment['state']} | {top_segment['remoteness_group']} | "
                f"{top_segment['speed_band']} | {top_segment['road_user_group']}"
            ),
            "period": "2024-Jan 2026",
            "dashboard_use": "Risk segment matrix and what-if panel",
        },
        {
            "finding": "Top latest risk segment fatalities",
            "value": int(top_segment["fatalities"]),
            "period": "2024-Jan 2026",
            "dashboard_use": "What-if base count",
        },
        {
            "finding": "Top day-time fatality cell",
            "value": f"{top_day_time['dayweek']} | {top_day_time['time_band']}",
            "period": "2024-Jan 2026",
            "dashboard_use": "Day x time heatmap annotation",
        },
        {
            "finding": "Top day-time fatality cell count",
            "value": int(top_day_time["fatalities"]),
            "period": "2024-Jan 2026",
            "dashboard_use": "Day x time heatmap annotation",
        },
    ]
)

print("EDA key findings:")
print(eda_key_findings.to_string(index=False))


## 10. Export Integrated Outputs


In [ ]:
master_path = OUTPUT_DIR / "master_dashboard.csv"
state_year_rates_path = OUTPUT_DIR / "state_year_rates.csv"
state_year_rates_recent_path = OUTPUT_DIR / "state_year_rates_recent_2024_2025.csv"
state_month_dashboard_path = OUTPUT_DIR / "state_month_dashboard.csv"
risk_segments_path = OUTPUT_DIR / "risk_segments_latest_2024_jan2026.csv"
day_time_path = OUTPUT_DIR / "day_time_heatmap_latest_2024_jan2026.csv"
road_user_speed_path = OUTPUT_DIR / "road_user_speed_latest_2024_jan2026.csv"
remoteness_speed_path = OUTPUT_DIR / "remoteness_speed_latest_2024_jan2026.csv"
annual_path = OUTPUT_DIR / "annual_fatalities.csv"
eda_findings_path = OUTPUT_DIR / "eda_key_findings.csv"

master.to_csv(master_path, index=False)
state_year_rates.to_csv(state_year_rates_path, index=False)
state_year_rates_recent_full.to_csv(state_year_rates_recent_path, index=False)
state_month_dashboard.to_csv(state_month_dashboard_path, index=False)
risk_segments_latest.to_csv(risk_segments_path, index=False)
day_time_heatmap_latest.to_csv(day_time_path, index=False)
road_user_speed_latest.to_csv(road_user_speed_path, index=False)
remoteness_speed_latest.to_csv(remoteness_speed_path, index=False)
annual_fatalities.to_csv(annual_path, index=False)
eda_key_findings.to_csv(eda_findings_path, index=False)

print("Exported files:")
for path in [
    master_path,
    state_year_rates_path,
    state_year_rates_recent_path,
    state_month_dashboard_path,
    risk_segments_path,
    day_time_path,
    road_user_speed_path,
    remoteness_speed_path,
    annual_path,
    eda_findings_path,
]:
    print(path)


## 11. Final Integration Validation


In [ ]:
checks = {
    "master_rows_equal_bitre_rows": len(master) == len(bitre),
    "master_has_no_duplicate_added_rows": len(master) == original_rows,
    "population_missing_only_2026_expected": int(master["population"].isna().sum()) == int(master["year"].eq(2026).sum()),
    "weather_available_matches_latest_window": int(master["weather_available"].eq("Yes").sum()) == int(master["analysis_window"].eq("Recent: 2024-Jan 2026").sum()),
    "state_year_rates_no_duplicate_keys": state_year_rates.duplicated(["state", "year"]).sum() == 0,
    "state_month_dashboard_no_duplicate_keys": state_month_dashboard.duplicated(["state", "year", "month"]).sum() == 0,
}

print("Final validation checks")
for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nMaster output shape:", master.shape)
print("State-year rates shape:", state_year_rates.shape)
print("State-month dashboard shape:", state_month_dashboard.shape)
